# Normalization and Split


Input: `data/features/features_processed.csv`  
Output: `data/splits/train.csv`, `val.csv`, `test.csv`

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)

input_path = os.path.join('..', 'data', 'features', 'features_processed.csv')
output_dir = os.path.join('..', 'data', 'splits')
os.makedirs(output_dir, exist_ok=True)

df = pd.read_csv(input_path)
print(f'{df.admissionid.nunique()} admissions, {len(df):,} rows')

In [ ]:
# Merge lengthgroup from cohort CSV — not present in features_processed.csv
cohort = pd.read_csv(os.path.join('..', 'data', 'cohort', 'sepsis3_cohort.csv'))
df = df.merge(
    cohort[['admissionid', 'lengthgroup']].drop_duplicates(),
    on='admissionid',
    how='left'
)
print(f'lengthgroup distribution:')
print(df['lengthgroup'].value_counts(dropna=False).head())

## Normalization



In [ ]:
# Source: Hilders et al. data_processing.ipynb normalization approach
age_map = {
    '18-39': -1.0, '40-49': -0.6667, '50-59': -0.3333,
    '60-69': 0.0, '70-79': 0.3333, '80+': 1.0,
}
df['admission_age_norm'] = df['agegroup'].map(age_map)

weight_mid = {
    '59-': 52.0, '60-69': 65.0, '70-79': 75.0, '80-89': 85.0,
    '90-99': 95.0, '100-109': 105.0, '110+': 115.0,
}
df['weight_mid'] = df['weightgroup'].map(weight_mid)
df['weight_null'] = np.where(df['weight_mid'].isna(), 1, 0)
df['weight_norm'] = (((df['weight_mid'].clip(55, 115) - 55) / 60) * 2 - 1).fillna(0.0)

length_mid = {
    '159-':    152.0,
    '160-169': 165.0,
    '170-179': 175.0,
    '180-189': 185.0,
    '190+':    195.0,
}
df['length_mid'] = df['lengthgroup'].map(length_mid)
df['height_null'] = np.where(df['length_mid'].isna(), 1, 0)
df['height_norm'] = (((df['length_mid'].clip(155, 195) - 155) / 40) * 2 - 1).fillna(0.0)
df.drop(columns=['lengthgroup', 'length_mid'], inplace=True)

df['gender'] = df['gender'].fillna(0).astype(int)
df.drop(columns=['agegroup', 'weightgroup', 'weight_mid'], inplace=True)

print(f'Columns after normalization ({len(df.columns)}):')
print(list(df.columns))

## Train / val / test split



In [4]:
# ClaudeAI. Prompt- Need to split cohorts into train/val/test.
ids = df['admissionid'].unique()
train_ids, temp = train_test_split(ids, test_size=0.30, random_state=42)
val_ids, test_ids = train_test_split(temp, test_size=0.50, random_state=42)

train = df[df['admissionid'].isin(train_ids)]
val   = df[df['admissionid'].isin(val_ids)]
test  = df[df['admissionid'].isin(test_ids)]

print(f'Train: {train.admissionid.nunique():,} admissions  {len(train):,} rows  ({len(train)/len(df)*100:.1f}%)')
print(f'Val:   {val.admissionid.nunique():,} admissions  {len(val):,} rows  ({len(val)/len(df)*100:.1f}%)')
print(f'Test:  {test.admissionid.nunique():,} admissions  {len(test):,} rows  ({len(test)/len(df)*100:.1f}%)')

print('\n90-day mortality by split:')
for name, d in [('Train', train), ('Val', val), ('Test', test)]:
    print(f'  {name}: {d["90day_mortality"].mean():.3f}')

Train: 1,850 admissions  114,740 rows  (70.4%)
Val:   397 admissions  23,890 rows  (14.7%)
Test:  397 admissions  24,381 rows  (15.0%)

90-day mortality by split:
  Train: 0.407
  Val: 0.335
  Test: 0.329


In [ ]:
train.to_csv(os.path.join(output_dir, 'train.csv'), index=False)
val.to_csv(os.path.join(output_dir, 'val.csv'),   index=False)
test.to_csv(os.path.join(output_dir, 'test.csv'),  index=False)

print(f'Saved to {output_dir}')
train.head()